<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/landingcollab/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Image Classification with WeightsLab

This notebook trains a small CNN on **MNIST** and instruments it with WeightsLab so every training signal is traced **back to the exact samples** producing it.

Most data problems (mislabels, outliers, class imbalance) stay invisible until your model tells you through the loss. By wrapping your training objects with `wl.watch_or_edit(...)`, WeightsLab records **per-sample** loss and metrics live, so you can rank the worst samples, spot bad labels, and curate the dataset **without restarting training**.

### What you'll do
1. Install WeightsLab.
2. Set every knob in one **config** dict (like a `config.yaml`).
3. Wrap the model, optimizer, dataloaders, loss, and metric with the SDK.
4. Train while per-sample signals are captured.
5. Rank the highest-loss samples inline, and (optionally) open the live **Weights Studio** UI.

## Setup

Install WeightsLab from PyPI. On Colab the free **T4 GPU** runtime is plenty for this demo (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [1]:
%pip install torch torchvision

In [2]:
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev6"

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple/
ERROR: Could not find a version that satisfies the requirement weightslab==1.3.3.dev6 (from versions: 0.0.0, 1.1.1, 1.1.2, 1.1.3.dev0, 1.1.3, 1.1.4, 1.1.5, 1.1.6, 1.1.7.dev3, 1.1.7, 1.1.8, 1.1.9, 1.2.0, 1.2.1.dev0, 1.2.1, 1.2.2, 1.2.3.dev0, 1.2.3, 1.2.4, 1.2.5, 1.2.6, 1.3.0.dev0, 1.3.0, 1.3.1.dev1, 1.3.1.dev2, 1.3.1.dev3, 1.3.1, 1.3.2.dev0, 1.3.2, 1.3.3.dev0, 1.3.3.dev1, 1.3.3.dev2, 1.3.3.dev3, 1.3.3.dev4, 1.3.3.dev5, 1.3.3, 1.9.1.dev1, 1.9.1.dev2, 1.9.1.dev3, 1.9.1.dev4, 20260427.dev1, 20260427.dev2, 20260605.dev1, 20260310110810, 20260310112657.dev2659810514, 20260310112936.dev2035376017, 20260310114507.dev36735734, 20260310122733.dev571756140, 20260310130048.dev500189762, 20260310142756.dev2389960633, 20260310172643.dev1790296826, 20260310184554.dev3572723963, 20260310190409.dev16171358, 20260311001628.dev82842326, 20260311001938.dev237981957, 20260311094631.dev2433054235, 20260311100556.dev2759443978, 20260

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [3]:
import os
import tempfile
import logging

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.examples.utils.baseline_models.pytorch.models import FashionCNN as CNN
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

15/07/2026-14:06:00.794 INFO:root:setup_logging: WeightsLab logging initialized - Log file: /tmp/tmpn4he5yzg/weightslab_logs/weightslab_20260715_140600.log
15/07/2026-14:06:00.796 INFO:weightslab:<module>: WeightsLab package initialized - Log level: INFO, Log to file: True
15/07/2026-14:06:00.797 INFO:weightslab:<module>: 
╭─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                     │
│  $$\      $$\           $$\           $$\         $$\               $$\                 $$\         │
│  $$ | $\  $$ |          \__|          $$ |        $$ |              $$ |                $$ |        │
│  $$ |$$$\ $$ | $$$$$$\  $$\  $$$$$$\  $$$$$$$\  $$$$$$\    $$$$$$$\ $$ |       $$$$$$\  $$$$$$$\    │
│  $$ $$ $$\$$ |$$  __$$\ $$ |$$  __$$\ $$  __$$\ \_$$  _|  $$  _____|$$ |       \____$$\ $$  __$$\   │
│  $$$$  _$$$$ |$$$$$$$$ |$$ |$$ /  $$ |$$ |  $$ | 

## 2. A dataset that carries sample identity

WeightsLab attributes every signal to a **sample id**. This thin wrapper around torchvision's MNIST returns `(image, id, label)` and attaches a virtual filepath per sample, so signals can be traced back to individual images in the UI.

In [4]:
class MNISTCustomDataset(Dataset):
    """MNIST that returns (image, index, label) and tracks a virtual filepath."""

    def __init__(self, root, train=True, download=False, transform=None):
        self.mnist = datasets.MNIST(root=root, train=train, download=download, transform=None)
        self.transform = transform
        self.train = train
        split = "train" if train else "test"
        self.filepaths = {}
        for idx in range(len(self.mnist)):
            label = self.mnist.targets[idx].item()
            self.filepaths[idx] = os.path.join(
                "MNIST", "processed", split, f"class_{label}", f"sample_{idx:05d}.pt"
            )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.transform:
            image = self.transform(image)
        return image, idx, label

## 3. Configuration

Every tunable lives here, in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets the Studio UI read (and live-edit) these values while training.

In [11]:
log_dir = tempfile.mkdtemp(prefix="weightslab_mnist_")

config = {
    # -- Experiment -------------------------------------------------------
    "experiment_name": "mnist_classification",    # name shown in Weights Studio
    "device": str(device),                        # "cuda" if a GPU is available, else "cpu"
    "root_log_dir": log_dir,                      # where signal history / dataframes are written
    "num_classes": 10,                            # MNIST digits 0-9

    # -- Training schedule ------------------------------------------------
    "training_steps_to_do": 8000,                 # total optimizer steps (raise for a longer live run)
    "eval_full_to_train_steps_ratio": 500,        # run a full eval every N steps
    "experiment_dump_to_train_steps_ratio": 250,   # dump model weights ratio
    "write_export_ratio": 1000,                   # export signal history + dataframe every N steps

    # Configure global dataframe storage
    "ledger_enable_flushing_threads": True,
    "ledger_enable_h5_persistence": True,
    "ledger_flush_max_rows": 1024,
    "ledger_flush_interval": 20.0,

    # -- Optimizer --------------------------------------------------------
    "learning_rate": 0.001,                        # Adam learning rate

    # -- Data loaders -----------------------------------------------------
    # One block per loader, keyed by loader_name. EVERY kwarg passed to
    # wl.watch_or_edit(..., flag="data") lives here so the wrap cell below stays
    # declarative — no dataloader settings hardcoded in the cells. Add a new
    # loader by adding another block here.
    "data": {
        "train_loader": {
            "batch_size": 128,                     # training batch size
            "shuffle": True,                      # shuffle each epoch
            "is_training": True,                  # marks this loader as the training split
            "compute_hash": False,                # skip per-sample content hashing (faster init)
            "preload_labels": True,               # preload labels into the ledger at startup
            "preload_metadata": False,            # load metadata lazily on first access
            "enable_h5_persistence": True,        # persist per-sample stats to the H5 store
        },
        "test_loader": {
            "batch_size": 128,                    # evaluation batch size
            "shuffle": False,                     # keep test order stable
            "is_training": False,                 # marks this loader as an eval split
            "compute_hash": False,
            "preload_labels": True,
            "preload_metadata": False,
            "enable_h5_persistence": True,
        },
    },

    # -- Services ---------------------------------------------------------
    "serving_grpc": True,                         # expose the gRPC backend for Weights Studio
    "serving_bore": True,                         # expose the bore tunnel for Weights Studio
}

wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)
print("Experiment logs ->", log_dir)

15/07/2026-14:23:06.251 INFO:weightslab.components.checkpoint_manager:__init__: CheckpointManager initialized at /tmp/weightslab_mnist_yonuysxx
15/07/2026-14:23:06.253 INFO:weightslab.src:watch_or_edit: Registered new checkpoint manager in ledger
Experiment logs -> /tmp/weightslab_mnist_yonuysxx


## 4. Wrap the training objects

This is the heart of WeightsLab. Each object is passed through `wl.watch_or_edit(...)` with a `flag` describing its role. The returned objects behave exactly like the originals, but now report their state and per-sample signals to WeightsLab.

In [6]:
data_root = os.path.join(log_dir, "data")
os.makedirs(data_root, exist_ok=True)

train_ds = MNISTCustomDataset(root=data_root, train=True, download=True,
                              transform=transforms.ToTensor())
test_ds = MNISTCustomDataset(root=data_root, train=False, download=True,
                             transform=transforms.ToTensor())

# Model + optimizer
model = wl.watch_or_edit(CNN().to(device), flag="model", device=device, compute_dependencies=True)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=config["learning_rate"]), flag="optimizer")

# Tracked dataloaders — all loader settings come from config["data"][<loader_name>],
# so nothing is hardcoded here. Unpack each block straight into the wrapper.
train_loader = wl.watch_or_edit(
    train_ds, flag="data", loader_name="train_loader",
    **config["data"]["train_loader"],
)
test_loader = wl.watch_or_edit(
    test_ds, flag="data", loader_name="test_loader",
    **config["data"]["test_loader"],
)

# Watched losses + metric (they log themselves per sample)
train_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                   flag="loss", signal_name="train-loss-CE", log=True)
test_criterion = wl.watch_or_edit(nn.CrossEntropyLoss(reduction="none"),
                                  flag="loss", signal_name="test-loss-CE", log=True)
metric = wl.watch_or_edit(
    Accuracy(task="multiclass", num_classes=config["num_classes"]).to(device),
    flag="metric", signal_name="metric-ACC", log=True)

15/07/2026-14:06:03.264 INFO:weightslab.backend.model_interface:__init__: Using checkpoint manager from ledger
15/07/2026-14:06:03.266 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loading checkpoint 124b3608e98f2d94...
15/07/2026-14:06:03.267 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Target: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:03.268 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Current: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:03.269 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Model architecture unchanged, using current model
15/07/2026-14:06:03.292 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [OK] Loaded weights from step 6750 with RNG state
15/07/2026-14:06:03.294 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Config unchanged, using current config
15/07/2026-14:06:03.296 INFO:weightslab.components.checkpoint_manager:load_ch

Initializing ledger for split 'train_loader': 100%|██████████| 60000/60000 [00:12<00:00, 4884.89it/s]

15/07/2026-14:06:15.708 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'train_loader' with 60000 samples.


15/07/2026-14:06:16.062 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 60000 samples → 60000 annotation rows.
15/07/2026-14:06:16.064 INFO:weightslab.data.h5_array_store:__init__: [H5ArrayStore] Initialized with cache limit: 2048MB
15/07/2026-14:06:17.387 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loading checkpoint 124b3608e98f2d94...
15/07/2026-14:06:17.389 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Target: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:17.390 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Current: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:17.393 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Model architecture unchanged, using current model
15/07/2026-14:06:17.393 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Config unchanged, using current config
15/07/2026-14:06:17.593 INFO:we

Initializing ledger for split 'test_loader': 100%|██████████| 10000/10000 [00:01<00:00, 7865.73it/s]

15/07/2026-14:06:19.213 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] Registering split 'test_loader' with 10000 samples.
15/07/2026-14:06:19.273 INFO:weightslab.data.dataframe_manager:register_split: [LedgeredDataFrameManager] After annotation expansion: 10000 samples → 10000 annotation rows.


15/07/2026-14:06:19.798 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loading checkpoint 124b3608e98f2d94...
15/07/2026-14:06:19.799 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Target: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:19.802 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  Current: HP=124b3608 MODEL=e98f2d94 DATA=9e70c5a5
15/07/2026-14:06:19.803 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Model architecture unchanged, using current model
15/07/2026-14:06:19.804 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [-] Config unchanged, using current config
15/07/2026-14:06:19.965 INFO:weightslab.components.checkpoint_manager:load_checkpoint:  [OK] Loaded data snapshot (70000 rows) with RNG state
15/07/2026-14:06:19.966 INFO:weightslab.components.checkpoint_manager:load_checkpoint: Loaded components: {'data'}
15/07/2026-14:06:20.277 INFO:weightslab.backend.dataloader_interface:

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab which phase it's in. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so the loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [7]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)

        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total_loss = loss.mean()
        total_loss.backward()
        optimizer.step()
    return total_loss.detach().cpu().item()


import time
def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    st = time.time()
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)

            loss = criterion(logits, labels, batch_ids=ids, preds=preds)
            losses += loss.mean()
            metric.update(logits, labels)

            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(
                preds_raw=logits, targets=labels, batch_ids=ids, preds=preds,
                signals={
                    "test_metric/Accuracy_per_sample": correct,
                    "test_metric/Inverse_Accuracy_per_sample": 1.0 - correct,
                },
            )
    print(f"Compute test in {time.time()-st}s")
    return (losses / n_batches).item(), (metric.compute() * 100).item()

## 6. Serve and train

`wl.serve(serving_grpc=True)` starts the background gRPC server (non-blocking) that Weights Studio connects to. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop, periodically evaluating and exporting signals.

In [8]:
wl.serve(serving_grpc=True, serving_bore=True)

15/07/2026-14:06:20.303 INFO:weightslab.trainer.trainer_services:_run_security_preflight: 

# #######################################
# #######################################
[gRPC] Security preflight checks:
	TLS: DISABLED (unencrypted traffic)
	Auth tokens: NONE configured
	! WARNING: GRPC_TLS_ENABLED=0. Traffic will be unencrypted. Use only for development.
	! WARNING: No GRPC_AUTH_TOKEN/GRPC_AUTH_TOKENS configured. Only transport-level trust (TLS/mTLS) will protect RPC access.
# #######################################
# #######################################

15/07/2026-14:06:20.305 INFO:weightslab.trainer.trainer_services:grpc_serve: [gRPC] Watchdogs disabled via WEIGHTSLAB_DISABLE_WATCHDOGS.
15/07/2026-14:06:20.306 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Thread callback started
15/07/2026-14:06:20.307 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Creating ThreadPoolExecutor with 6 worker threads (n_workers_grpc=None, m

[PreviewCache]:   0%|          | 0/2000 [00:00<?, ?sample/s]

15/07/2026-14:06:20.742 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Servicer added
15/07/2026-14:06:20.750 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Attempting to bind to 0.0.0.0:50051
15/07/2026-14:06:20.753 WARNING:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] TLS disabled; using insecure transport on 0.0.0.0:50051
15/07/2026-14:06:20.756 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Port 50051 bound successfully.
15/07/2026-14:06:20.763 INFO:weightslab.trainer.trainer_services:serving_thread_callback: [gRPC] Server started and listening on 0.0.0.0:50051


[PreviewCache]:  14%|█▎        | 270/2000 [00:00<00:02, 753.49sample/s]

 Backend exposed via bore at: bore.pub:11217
 On your local machine (Docker running), run:
     weightslab ui launch
     weightslab tunnel bore.pub:11217
15/07/2026-14:06:21.139 INFO:weightslab.backend.cli:cli_serve: cli_thread_started


'bore.pub:11217'

In [10]:
wl.start_training(timeout=3)

steps = 6000
eval_ratio = config["eval_full_to_train_steps_ratio"]
export_ratio = config["write_export_ratio"]
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(range(steps), desc="Training")
for step in pbar:
    age = model.get_age() if hasattr(model, "get_age") else step

    train_loss = train(train_loader, model, optimizer, train_criterion, device)

    if age == 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)

    postfix = {"loss": f"{train_loss:.3f}"}
    if test_acc is not None:
        postfix["test_acc"] = f"{test_acc:.1f}%"
    pbar.set_postfix(postfix)

wl.write_history()
wl.write_dataframe()
print("Training complete. Logs at:", log_dir)

15/07/2026-14:15:41.616 INFO:weightslab.src:start_training: Starting WeightsLab training mode with a timeout of 3 seconds.
15/07/2026-14:15:44.618 INFO:weightslab.components.global_monitoring:resume: 
Attempting to resume training...
15/07/2026-14:15:44.750 INFO:weightslab.components.global_monitoring:resume: Hashes by module: ['124b3608', 'e2c168e8', '9e70c5a5']
15/07/2026-14:15:44.751 INFO:weightslab.components.global_monitoring:resume: Resuming training now...
15/07/2026-14:15:44.752 INFO:weightslab.components.global_monitoring:resume: Hashes by module on resume: ['124b3608', 'e2c168e8', '9e70c5a5']
15/07/2026-14:15:44.754 INFO:weightslab.components.global_monitoring:resume: 
Training resumed as modules hashes have been computed: ['124b3608', 'e2c168e8', '9e70c5a5'].


Training:   0%|          | 0/6000 [00:00<?, ?it/s]

15/07/2026-14:15:51.755 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_015000.pt
15/07/2026-14:15:54.136 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
Compute test in 9.581607103347778s
15/07/2026-14:16:13.098 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_015250.pt
15/07/2026-14:16:14.780 INFO:weightslab.trainer.services.data_service:GetDataSplits: GetDataSplits returning: ['test_loader', 'train_loader']
15/07/2026-14:16:15.921 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
15/07/2026-14:16:21.044 WARNING:weightslab.trainer.services.data_service:_slowUpdateInternals: [LockWatchdog] _

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15/07/2026-14:17:43.085 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
15/07/2026-14:17:47.266 WARNING:weightslab.trainer.services.data_service:_slowUpdateInternals: [LockWatchdog] _update_lock[_slowUpdateInternals]   RELEASED by WL-ViewRefresh                 held 1029.7 ms ← SLOW
15/07/2026-14:17:49.998 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_016500.pt
15/07/2026-14:17:53.627 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
Compute test in 9.54624342918396s
15/07/2026-14:18:11.743 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_016750.pt
15/07/2026-14:18:14.452 INFO:weightslab.co

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15/07/2026-14:19:58.591 WARNING:weightslab.trainer.services.experiment_service:_get_latest_logger_data_impl: get_and_clear_queue() took 714.1ms (slow — possible lock contention)
15/07/2026-14:20:01.175 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
15/07/2026-14:20:07.495 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_018500.pt


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

15/07/2026-14:20:12.909 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
Compute test in 9.242669343948364s
15/07/2026-14:20:31.291 INFO:weightslab.components.checkpoint_manager:save_model_checkpoint: Saved model checkpoint: 124b3608e2c168e89e70c5a5_step_018750.pt
15/07/2026-14:20:32.245 WARNING:weightslab.trainer.services.experiment_service:_get_latest_logger_data_impl: get_and_clear_queue() took 723.2ms (slow — possible lock contention)
15/07/2026-14:20:34.664 INFO:weightslab.components.checkpoint_manager:save_logger_snapshot: Saved logger snapshot: /tmp/weightslab_mnist_kg8215pb/checkpoints/loggers/loggers.manifest.json (1 chunks)
15/07/2026-14:20:37.308 WARNING:weightslab.trainer.services.data_service:_slowUpdateInternals: [LockWatchdog] _update_lock[_slowUpdateInternals]   RELEASED by WL-ViewRefresh                 held 1075.4 ms ← SLOW
15/07/2026-14:20:43.

## 9. See it live in Weights Studio

Everything above ran headless. The real payoff is the **Weights Studio** UI, where you browse the highest-loss images, filter by class, and curate the dataset mid-training.

Studio runs as a local Docker stack (a static frontend + an Envoy gRPC-Web proxy). **Colab has no Docker daemon**, so you don't run the UI *inside* Colab - you run Studio on your own machine and point it at this notebook's backend using the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab

# Terminal 1 - launch the UI (plaintext HTTP, the default)
weightslab ui launch                       # opens http://localhost:5173

# Terminal 2 - bridge the Colab backend to localhost:50051
weightslab tunnel bore.pub:12345           # the host:port printed in Section 6
```

Then open **http://localhost:5173** - Studio connects through your local Envoy -> `weightslab tunnel` -> `bore.pub` -> this Colab backend, and you watch training stream live.

> Note: keep it plaintext end-to-end (the default `weightslab ui launch`, raw-TCP `bore`) so gRPC's HTTP/2 frames pass through untouched.

Prefer to keep it all local? Run this same example on your own machine (`weightslab start example --cls`) and launch the UI next to it - no tunnel required.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>